# MatMul OpenCL Benchmark Dashboard

Ce notebook teste plusieurs fois les versions `matmul_coalsced.py`, `matmul_uncoalsced.py` et `matmul_block.py` **sans modifier leur code source**.

Il propose deux modes:
1. Bench parametrique (meme kernels, dimensions/local sizes/devices variables).
2. Bench script-level (execution directe des 3 scripts originaux via `subprocess`).

Visualisations incluses:
- Courbe GPU en fonction de la dimension.
- Comparaison des devices pour une meme dimension.
- Stabilite run-by-run (10 executions).
- Heatmaps analytiques de performance.

In [ ]:
# Si besoin, decommenter la ligne suivante:
# !pip install pyopencl pandas matplotlib seaborn

from pathlib import Path
from time import perf_counter
import os
import re
import sys
import subprocess

import numpy as np
import pyopencl as cl
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

In [ ]:
BASE_DIR = Path.cwd()
candidates = [
    BASE_DIR,
    BASE_DIR / "opencl_examples",
    BASE_DIR / "TP-CODESIGN" / "opencl_examples",
    BASE_DIR.parent,
]

OPENCL_DIR = None
for c in candidates:
    if (c / "prof_files" / "matmul_coalsced.py").exists():
        OPENCL_DIR = c
        break

assert OPENCL_DIR is not None, "Dossier opencl_examples/prof_files introuvable."
EXAMPLE_DIR = OPENCL_DIR / "prof_files"
BENCH_DIR = OPENCL_DIR / "tp0_benchmark" if (OPENCL_DIR / "tp0_benchmark").exists() else OPENCL_DIR

SCRIPT_MAP = {
    "matmul_coalsced": EXAMPLE_DIR / "matmul_coalsced.py",
    "matmul_uncoalsced": EXAMPLE_DIR / "matmul_uncoalsced.py",
    "matmul_block": EXAMPLE_DIR / "matmul_block.py",
}

KERNEL_MAP = {
    "matmul_coalsced": EXAMPLE_DIR / "C_elem_ji.cl",
    "matmul_uncoalsced": EXAMPLE_DIR / "C_elem_ij.cl",
    "matmul_block": EXAMPLE_DIR / "C_block_form.cl",
}

import sys
sys.path.insert(0, str(EXAMPLE_DIR))
try:
    from definitions import AVAL, BVAL, COUNT
except Exception:
    AVAL, BVAL, COUNT = 3.257, 5.723, 20

print("OPENCL_DIR =", OPENCL_DIR)
print("EXAMPLE_DIR =", EXAMPLE_DIR)
print("BENCH_DIR =", BENCH_DIR)
print("AVAL, BVAL, COUNT =", AVAL, BVAL, COUNT)


In [ ]:
def discover_device_entries():
    entries = []
    for p_idx, platform in enumerate(cl.get_platforms()):
        for d_idx, device in enumerate(platform.get_devices()):
            entries.append({
                "platform_index": p_idx,
                "device_index": d_idx,
                "platform_name": platform.name.strip(),
                "device_name": device.name.strip(),
                "device_type": cl.device_type.to_string(device.type),
                "global_mem_gb": float(device.global_mem_size / (2**30)),
                "max_compute_units": int(device.max_compute_units),
                "ctx_selector": f"{p_idx}:{d_idx}",
                "device": device,
            })
    return entries

DEVICE_ENTRIES = discover_device_entries()
devices_df = pd.DataFrame([{k: v for k, v in e.items() if k != "device"} for e in DEVICE_ENTRIES])

if devices_df.empty:
    raise RuntimeError("Aucun device OpenCL detecte.")

display(devices_df)

In [ ]:
def _load_kernel_source(variant: str, local_size: int) -> str:
    src = KERNEL_MAP[variant].read_text(encoding="utf-8")
    if variant == "matmul_block":
        src = re.sub(r"#define\s+blksz\s+\d+", f"#define blksz {local_size}", src, count=1)
    return src


def _build_program(context: cl.Context, variant: str, local_size: int):
    src = _load_kernel_source(variant, local_size)
    return cl.Program(context, src).build()


def run_single_config(device_entry, variant: str, N: int, local_size: int, repeats: int = 10, launches_per_run: int = COUNT):
    rows = []
    expected_c00 = float(N) * float(AVAL) * float(BVAL)

    if N % local_size != 0:
        for run_id in range(1, repeats + 1):
            rows.append({
                "variant": variant,
                "N": N,
                "local_size": local_size,
                "run_id": run_id,
                "status": "skip_incompatible_global_local",
                "error": f"N ({N}) must be divisible by local_size ({local_size})",
                "time_s": np.nan,
                "mflops": np.nan,
                "c00": np.nan,
                "c00_abs_error": np.nan,
                "platform_name": device_entry["platform_name"],
                "device_name": device_entry["device_name"],
                "device_type": device_entry["device_type"],
                "ctx_selector": device_entry["ctx_selector"],
            })
        return rows

    try:
        context = cl.Context(devices=[device_entry["device"]])
        queue = cl.CommandQueue(context)

        size = N * N
        h_a = np.full(size, AVAL, dtype=np.float32)
        h_b = np.full(size, BVAL, dtype=np.float32)
        h_c = np.empty(size, dtype=np.float32)

        d_a = cl.Buffer(context, cl.mem_flags.READ_ONLY | cl.mem_flags.COPY_HOST_PTR, hostbuf=h_a)
        d_b = cl.Buffer(context, cl.mem_flags.READ_ONLY | cl.mem_flags.COPY_HOST_PTR, hostbuf=h_b)
        d_c = cl.Buffer(context, cl.mem_flags.WRITE_ONLY, size=h_c.nbytes)

        program = _build_program(context, variant, local_size)
        mmul = program.mmul

        if variant == "matmul_block":
            mmul.set_scalar_arg_dtypes([np.uint32, None, None, None, None, None])
            a_block = cl.LocalMemory(np.dtype(np.float32).itemsize * local_size * local_size)
            b_block = cl.LocalMemory(np.dtype(np.float32).itemsize * local_size * local_size)
        else:
            mmul.set_scalar_arg_dtypes([np.int32, None, None, None])
            a_block = None
            b_block = None

        for run_id in range(1, repeats + 1):
            start = perf_counter()
            for _ in range(launches_per_run):
                if variant == "matmul_block":
                    mmul(queue, (N, N), (local_size, local_size), np.uint32(N), d_a, d_b, d_c, a_block, b_block)
                else:
                    mmul(queue, (N, N), (local_size, local_size), np.int32(N), d_a, d_b, d_c)
            queue.finish()
            elapsed = perf_counter() - start

            cl.enqueue_copy(queue, h_c, d_c).wait()
            c00 = float(h_c[0])
            c00_abs_error = abs(c00 - expected_c00)
            mflops = (2.0 * launches_per_run * (N ** 3)) / (1_000_000.0 * elapsed)

            rows.append({
                "variant": variant,
                "N": N,
                "local_size": local_size,
                "run_id": run_id,
                "status": "ok",
                "error": "",
                "time_s": elapsed,
                "mflops": mflops,
                "c00": c00,
                "c00_abs_error": c00_abs_error,
                "platform_name": device_entry["platform_name"],
                "device_name": device_entry["device_name"],
                "device_type": device_entry["device_type"],
                "ctx_selector": device_entry["ctx_selector"],
            })

    except Exception as exc:
        for run_id in range(1, repeats + 1):
            rows.append({
                "variant": variant,
                "N": N,
                "local_size": local_size,
                "run_id": run_id,
                "status": "error",
                "error": str(exc),
                "time_s": np.nan,
                "mflops": np.nan,
                "c00": np.nan,
                "c00_abs_error": np.nan,
                "platform_name": device_entry["platform_name"],
                "device_name": device_entry["device_name"],
                "device_type": device_entry["device_type"],
                "ctx_selector": device_entry["ctx_selector"],
            })

    return rows


def run_campaign(device_entries, variants, dims, local_sizes, repeats=10, launches_per_run=COUNT):
    all_rows = []
    total = len(device_entries) * len(variants) * len(dims) * len(local_sizes)
    done = 0

    for entry in device_entries:
        for variant in variants:
            for N in dims:
                for local_size in local_sizes:
                    done += 1
                    print(f"[{done:04d}/{total:04d}] device={entry['device_name']} | variant={variant} | N={N} | local={local_size}")
                    all_rows.extend(run_single_config(
                        entry,
                        variant=variant,
                        N=N,
                        local_size=local_size,
                        repeats=repeats,
                        launches_per_run=launches_per_run,
                    ))

    return pd.DataFrame(all_rows)

In [ ]:
# ---------- Configuration principale ----------
VARIANTS = ["matmul_uncoalsced", "matmul_coalsced", "matmul_block"]
DIMS = [256, 512, 1024]
LOCAL_SIZES = [4, 8, 16, 32]
REPEATS = 10                    # comportement run-by-run
LAUNCHES_PER_RUN = COUNT        # comme les scripts originaux
DEVICE_TYPE_FILTER = None       # None, "GPU", "CPU"

selected_entries = DEVICE_ENTRIES
if DEVICE_TYPE_FILTER is not None:
    selected_entries = [e for e in DEVICE_ENTRIES if DEVICE_TYPE_FILTER.upper() in e["device_type"].upper()]

print(f"Devices selectionnes: {len(selected_entries)}")
for e in selected_entries:
    print(f" - {e['device_name']} ({e['device_type']})")

In [ ]:
bench_df = run_campaign(
    selected_entries,
    variants=VARIANTS,
    dims=DIMS,
    local_sizes=LOCAL_SIZES,
    repeats=REPEATS,
    launches_per_run=LAUNCHES_PER_RUN,
)

print("\nStatus count:")
print(bench_df["status"].value_counts(dropna=False))

display(bench_df.head())

# Sauvegarde CSV pour reutilisation
csv_path = BENCH_DIR / "matmul_benchmark_results.csv"
bench_df.to_csv(csv_path, index=False)
print("CSV sauvegarde:", csv_path)


In [ ]:
ok_df = bench_df[bench_df["status"] == "ok"].copy()

# 1) Courbe GPU: performance selon la dimension
gpu_df = ok_df[ok_df["device_type"].str.contains("GPU", case=False, na=False)].copy()

if gpu_df.empty:
    print("Aucune mesure GPU valide pour ce filtre.")
else:
    agg_gpu = gpu_df.groupby(["variant", "device_name", "N"], as_index=False)["mflops"].mean()
    plt.figure(figsize=(11, 6))
    sns.lineplot(
        data=agg_gpu,
        x="N",
        y="mflops",
        hue="device_name",
        style="variant",
        markers=True,
        dashes=False,
    )
    plt.title("GPU vs Dimension (MFLOPS moyen)")
    plt.xlabel("Dimension N")
    plt.ylabel("MFLOPS moyen")
    plt.tight_layout()
    plt.show()

In [ ]:
# 2) Valeur donnee par chaque device pour une meme dimension
TARGET_DIM = max(DIMS)
same_dim_df = ok_df[ok_df["N"] == TARGET_DIM].copy()

if same_dim_df.empty:
    print(f"Aucune mesure valide pour N={TARGET_DIM}.")
else:
    agg_same_dim = same_dim_df.groupby(["device_name", "variant"], as_index=False)["mflops"].mean()
    plt.figure(figsize=(12, 6))
    sns.barplot(data=agg_same_dim, x="device_name", y="mflops", hue="variant")
    plt.title(f"Comparaison devices a N={TARGET_DIM} (MFLOPS moyen)")
    plt.xlabel("Device")
    plt.ylabel("MFLOPS moyen")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
# 3) Comportement de chaque device sur 10 executions du meme script
FOCUS_VARIANT = "matmul_coalsced"
FOCUS_N = max(DIMS)
FOCUS_LOCAL_SIZE = 16

trend_df = ok_df[(ok_df["variant"] == FOCUS_VARIANT) & (ok_df["N"] == FOCUS_N) & (ok_df["local_size"] == FOCUS_LOCAL_SIZE)].copy()

if trend_df.empty:
    print("Aucune donnee pour le filtre trend. Modifie FOCUS_VARIANT/FOCUS_N/FOCUS_LOCAL_SIZE.")
else:
    plt.figure(figsize=(12, 6))
    sns.lineplot(
        data=trend_df,
        x="run_id",
        y="mflops",
        hue="device_name",
        marker="o",
    )
    plt.title(f"Stabilite run-by-run ({FOCUS_VARIANT}, N={FOCUS_N}, local={FOCUS_LOCAL_SIZE})")
    plt.xlabel("Numero de run")
    plt.ylabel("MFLOPS")
    plt.xticks(sorted(trend_df["run_id"].unique()))
    plt.tight_layout()
    plt.show()

In [ ]:
# 4) Vision analytique supplementaire: heatmaps
if ok_df.empty:
    print("Pas de mesures valides pour la heatmap.")
else:
    mean_df = ok_df.groupby(["variant", "device_name", "N"], as_index=False)["mflops"].mean()

    variants_present = list(mean_df["variant"].unique())
    fig, axes = plt.subplots(1, len(variants_present), figsize=(6 * len(variants_present), 5), squeeze=False)

    for ax, variant in zip(axes[0], variants_present):
        sub = mean_df[mean_df["variant"] == variant]
        pivot = sub.pivot(index="device_name", columns="N", values="mflops")
        sns.heatmap(pivot, annot=True, fmt=".1f", cmap="YlGnBu", ax=ax)
        ax.set_title(f"{variant} - MFLOPS moyen")
        ax.set_xlabel("Dimension N")
        ax.set_ylabel("Device")

    plt.tight_layout()
    plt.show()

## Option: lancer directement les 3 scripts originaux

Cette section execute `matmul_coalsced.py`, `matmul_uncoalsced.py`, `matmul_block.py` tels quels via `subprocess` (toujours sans modifier leur code).

Notes:
- Les scripts imposent `N=2048` et `COUNT=20` en interne.
- On passe `local_size` via `stdin`.
- On force le device via `PYOPENCL_CTX` (format `platform_idx:device_idx`).

In [ ]:
PERF_RE = re.compile(r"([0-9]+(?:\.[0-9]+)?)\s+seconds at\s+([0-9]+(?:\.[0-9]+)?)\s+MFLOPS")


def run_script_once(script_path: Path, local_size: int, ctx_selector: str, timeout_s: int = 3600):
    env = os.environ.copy()
    env["PYOPENCL_CTX"] = ctx_selector

    proc = subprocess.run(
        [sys.executable, str(script_path)],
        input=f"{local_size}\n",
        text=True,
        capture_output=True,
        cwd=str(EXAMPLE_DIR),
        env=env,
        timeout=timeout_s,
    )

    stdout = proc.stdout or ""
    stderr = proc.stderr or ""
    match = PERF_RE.search(stdout)

    if proc.returncode == 0 and match:
        time_s = float(match.group(1))
        mflops = float(match.group(2))
        status = "ok"
        err = ""
    else:
        time_s = np.nan
        mflops = np.nan
        status = "error"
        err = (stderr.strip() or stdout[-1000:]).strip()

    return {
        "status": status,
        "error": err,
        "time_s": time_s,
        "mflops": mflops,
        "returncode": proc.returncode,
    }


def run_original_scripts_campaign(device_entries, script_map, local_size=16, repeats=10):
    rows = []
    total = len(device_entries) * len(script_map) * repeats
    done = 0

    for entry in device_entries:
        for script_name, script_path in script_map.items():
            for run_id in range(1, repeats + 1):
                done += 1
                print(f"[{done:04d}/{total:04d}] script={script_name} | device={entry['device_name']} | run={run_id}")
                out = run_script_once(script_path, local_size=local_size, ctx_selector=entry["ctx_selector"])
                rows.append({
                    "script": script_name,
                    "local_size": local_size,
                    "run_id": run_id,
                    "platform_name": entry["platform_name"],
                    "device_name": entry["device_name"],
                    "device_type": entry["device_type"],
                    "ctx_selector": entry["ctx_selector"],
                    **out,
                })

    return pd.DataFrame(rows)


RUN_ORIGINAL_SCRIPTS = True
SCRIPT_LOCAL_SIZE = 16
SCRIPT_REPEATS = 10
SCRIPT_DEVICE_TYPE_FILTER = "GPU"  # None, "GPU", "CPU"

if RUN_ORIGINAL_SCRIPTS:
    script_entries = DEVICE_ENTRIES
    if SCRIPT_DEVICE_TYPE_FILTER is not None:
        script_entries = [e for e in DEVICE_ENTRIES if SCRIPT_DEVICE_TYPE_FILTER.upper() in e["device_type"].upper()]

    script_df = run_original_scripts_campaign(
        script_entries,
        SCRIPT_MAP,
        local_size=SCRIPT_LOCAL_SIZE,
        repeats=SCRIPT_REPEATS,
    )

    display(script_df.head())
    print(script_df["status"].value_counts(dropna=False))

    script_csv = BENCH_DIR / "matmul_original_scripts_results.csv"
    script_df.to_csv(script_csv, index=False)
    print("CSV sauvegarde:", script_csv)
else:
    script_df = pd.DataFrame()
    print("RUN_ORIGINAL_SCRIPTS=False -> section sautee.")


In [ ]:
if script_df.empty:
    print("Pas de donnees script-level. Active RUN_ORIGINAL_SCRIPTS pour generer ce graphe.")
else:
    script_ok = script_df[script_df["status"] == "ok"].copy()
    if script_ok.empty:
        print("Aucune mesure script-level valide.")
    else:
        plt.figure(figsize=(12, 6))
        sns.lineplot(
            data=script_ok,
            x="run_id",
            y="mflops",
            hue="device_name",
            style="script",
            markers=True,
            dashes=False,
        )
        plt.title("Scripts originaux: stabilite sur 10 executions")
        plt.xlabel("Numero de run")
        plt.ylabel("MFLOPS")
        plt.tight_layout()
        plt.show()